# 07 — Embeddings, Similarity, and Vector Search

**Network LLM Engineering — Part II — Knowledge and Context**

### Learning goals
- Understand embedding vectors and similarity
- Build a small semantic search index
- Know when dense search fails

In [ ]:
%pip install -q transformers==5.14.1 datasets==5.0.1 accelerate==1.14.0 peft==0.20.0 trl==1.10.0 sentence-transformers==5.7.0 pandas matplotlib scikit-learn requests jsonschema

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineering_course")]:
        if (p / "data" / "glossary.csv").exists():
            return p
    raise FileNotFoundError("Run from the extracted network_llm_engineering_course folder.")

ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## Why embeddings?

An embedding maps text into a vector so semantically related texts are near one another.
The generative LLM is **not** the same component as the embedding model.

Networking retrieval benefits from both:
- exact identifiers: `BGP-3-NOTIFICATION`, `Ethernet1/49`, `RFC 4271`,
- semantic paraphrases: "peer never comes up" ~ "BGP session fails to establish".

In [ ]:
import json, numpy as np
from sentence_transformers import SentenceTransformer

docs = [json.loads(x) for x in open(DATA/"mini_network_knowledge.jsonl", encoding="utf-8")]
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
D = embedder.encode([d["text"] for d in docs], normalize_embeddings=True)

def search(q, k=3):
    qv = embedder.encode([q], normalize_embeddings=True)[0]
    scores = D @ qv
    idx = np.argsort(-scores)[:k]
    return [(float(scores[i]), docs[i]) for i in idx]

for score, d in search("small packets pass but large packets fail through tunnel"):
    print(round(score,3), d["topic"], "-", d["text"][:120])

## Dense search limitations

Embeddings can miss exact operational details. A semantic retriever may consider two error messages similar even when one character changes
the diagnosis. This is why **hybrid retrieval** (BM25/keyword + vectors) is often a better network design.

### Exercise

Search for:
1. `EVPN Type 2`
2. `remote leaf cannot reach host`
3. `CRC`

Which queries need semantic matching and which benefit from exact-term retrieval?